# 04 — Prediction Models

Walk-forward evaluation of all prediction models (base and regime variants).

**STOP POINT**: After linear model, confirm leakage test passes and inspect walk-forward output.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.utils.seeds import set_all_seeds
set_all_seeds()
%matplotlib inline

## 1. Load Panel and Generate Folds

In [ ]:
from src.data.panel import build_panel
from src.evaluation.walk_forward import generate_folds, assert_no_leakage

panel = build_panel(force_refresh=False)
print(f'Panel shape: {panel.shape}')

dates = panel.index.get_level_values('date').unique()
folds = generate_folds(dates, initial_train_years=5, test_months=1, refit_cadence_months=12)
assert_no_leakage(folds, panel)
print(f'{len(folds)} folds, leakage check PASSED')

## 2. Linear Regression (STOP POINT — validate pipeline end-to-end)

In [ ]:
from src.models.linear import LinearReturnModel
from src.evaluation.walk_forward import run_walk_forward
from src.utils.io import save_parquet
from pathlib import Path

feature_cols = [c for c in panel.columns if c != 'target']
linear_model = LinearReturnModel()
linear_results = run_walk_forward(linear_model, panel, folds, feature_cols)

print(f'Linear base results: {linear_results.shape}')
print(linear_results.head(10))

save_parquet(linear_results, Path('../results/predictions/linear_base.parquet'))

## 3. Logistic Direction Model

In [ ]:
from src.models.linear import LogisticDirectionModel

logistic_model = LogisticDirectionModel()
logistic_results = run_walk_forward(logistic_model, panel, folds, feature_cols)
save_parquet(logistic_results, Path('../results/predictions/logistic_base.parquet'))
print(f'Logistic base: {logistic_results.shape}')

## 4. PCA Factor Model

In [ ]:
from src.models.pca_factor import PCAFactorModel

pca_model = PCAFactorModel(n_components=5)
pca_results = run_walk_forward(pca_model, panel, folds, feature_cols)
save_parquet(pca_results, Path('../results/predictions/pca_base.parquet'))
print(f'PCA explained variance (last fold): {pca_model.cumulative_variance_explained_:.3f}')

## 5. LSTM

In [ ]:
from src.models.lstm import LSTMReturnModel

lstm_model = LSTMReturnModel(seq_len=60, hidden_size=64, n_layers=2, epochs=50)
lstm_results = run_walk_forward(lstm_model, panel, folds, feature_cols)
save_parquet(lstm_results, Path('../results/predictions/lstm_base.parquet'))
print(f'LSTM base: {lstm_results.shape}')

## 6. Regime Variants (requires HMM probs)

In [ ]:
from src.utils.io import load_parquet
hmm_probs = load_parquet('../data/regimes/hmm_probs.parquet')

if hmm_probs is not None:
    from src.experiments.run_all import _append_regime_features
    panel_regime = _append_regime_features(panel, hmm_probs)
    feat_regime = [c for c in panel_regime.columns if c != 'target']

    for ModelCls, name in [(LinearReturnModel, 'linear'), (LogisticDirectionModel, 'logistic'),
                            (PCAFactorModel, 'pca')]:
        m = ModelCls()
        res = run_walk_forward(m, panel_regime, folds, feat_regime)
        save_parquet(res, Path(f'../results/predictions/{name}_regime.parquet'))
        print(f'{name}_regime: {res.shape}')
else:
    print('HMM probs not found -- run notebook 03 first.')

## 7. Compare Prediction Errors

In [ ]:
from pathlib import Path
pred_files = sorted(Path('../results/predictions').glob('*.parquet'))

fig, ax = plt.subplots(figsize=(12, 5))
for pf in pred_files:
    df = load_parquet(pf)
    if df is not None and 'prediction' in df.columns and 'target' in df.columns:
        err = (df['prediction'] - df['target']).dropna()
        ax.hist(err, bins=60, alpha=0.4, label=pf.stem, density=True)

ax.set_xlabel('Prediction Error')
ax.set_title('Per-Model Prediction Error Distribution')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()